In [1]:
import os
import pandas as pd
import numpy as np

In [3]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

RAW_DATA_PATH = "../data/raw"

# Saare 8 raw CSVs load karna
customers = pd.read_csv(os.path.join(RAW_DATA_PATH, "olist_customers_dataset.csv"))
geolocation = pd.read_csv(os.path.join(RAW_DATA_PATH, "olist_geolocation_dataset.csv"))
order_items = pd.read_csv(os.path.join(RAW_DATA_PATH, "olist_order_items_dataset.csv"))
payments = pd.read_csv(os.path.join(RAW_DATA_PATH, "olist_order_payments_dataset.csv"))
reviews = pd.read_csv(os.path.join(RAW_DATA_PATH, "olist_order_reviews_dataset.csv"))
orders = pd.read_csv(os.path.join(RAW_DATA_PATH, "olist_orders_dataset.csv"))
products = pd.read_csv(os.path.join(RAW_DATA_PATH, "olist_products_dataset.csv"))
sellers = pd.read_csv(os.path.join(RAW_DATA_PATH, "olist_sellers_dataset.csv"))
category_translation = pd.read_csv(os.path.join(RAW_DATA_PATH, "product_category_name_translation.csv"))

datasets = {
    "customers": customers,
    "geolocation": geolocation,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "orders": orders,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation
}

# Row count aur column count ka audit summary
shape_summary = pd.DataFrame([
    {"dataset": name, "total_rows": df.shape[0], "total_columns": df.shape[1]}
    for name, df in datasets.items()
])

display(shape_summary)

,dataset,total_rows,total_columns
0,customers,99441,5
1,geolocation,1000163,5
2,order_items,112650,7
3,payments,103886,5
4,reviews,99224,7
5,orders,99441,8
6,products,32951,9
7,sellers,3095,4
8,category_translation,71,2


In [4]:

# Step 2: Comprehensive Missing Values Audit across all Datasets

missing_records = []

for name, df in datasets.items():
    null_counts = df.isnull().sum()
    null_percentages = (null_counts / len(df)) * 100
    
    # Sirf un columns ko filter karna jinme missing values hain (> 0)
    missing_cols = null_counts[null_counts > 0]
    
    for col in missing_cols.index:
        missing_records.append({
            "dataset": name,
            "column": col,
            "missing_count": null_counts[col],
            "missing_pct": round(null_percentages[col], 2),
            "data_type": str(df[col].dtype)
        })

missing_df = pd.DataFrame(missing_records)

# Agar missing values mili toh print karo
if not missing_df.empty:
    missing_summary = missing_df.sort_values(by=["dataset", "missing_pct"], ascending=[True, False])
    display(missing_summary)
else:
    print("Kisi bhi dataset mein koi missing value nahi mili!")

,dataset,column,missing_count,missing_pct,data_type
4,orders,order_delivered_customer_date,2965,2.98,str
3,orders,order_delivered_carrier_date,1783,1.79,str
2,orders,order_approved_at,160,0.16,str
5,products,product_category_name,610,1.85,str
6,products,product_name_lenght,610,1.85,float64
7,products,product_description_lenght,610,1.85,float64
8,products,product_photos_qty,610,1.85,float64
9,products,product_weight_g,2,0.01,float64
10,products,product_length_cm,2,0.01,float64
11,products,product_height_cm,2,0.01,float64


In [ ]:
# Step 3: Duplicate Records & Primary Key Uniqueness Audit

# 1. Full Row Duplicates Audit

duplicate_rows = []
for name, df in datasets.items():
    dup_count = df.duplicated().sum()
    duplicate_rows.append({
        "dataset": name,
        "total_rows": len(df),
        "duplicate_rows": dup_count,
        "duplicate_pct": round((dup_count / len(df)) * 100, 4)
    })

dup_summary_df = pd.DataFrame(duplicate_rows)
print(" Full Row Duplicates Audit ")
display(dup_summary_df)

# 2. Primary Key Uniqueness Audit
pk_definitions = {
    "customers": ["customer_id"],
    "orders": ["order_id"],
    "order_items": ["order_id", "order_item_id"],  # Composite primary key
    "payments": ["order_id", "payment_sequential"],  # Composite key
    "reviews": ["review_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "geolocation": ["geolocation_zip_code_prefix"]  # Many-to-many expected, check uniqueness
}

pk_audit = []
for dataset_name, pk_cols in pk_definitions.items():
    df = datasets[dataset_name]
    total_records = len(df)
    unique_keys = df.drop_duplicates(subset=pk_cols).shape[0]
    duplicate_keys = total_records - unique_keys
    
    pk_audit.append({
        "dataset": dataset_name,
        "key_columns": ", ".join(pk_cols),
        "total_records": total_records,
        "unique_keys": unique_keys,
        "duplicate_keys": duplicate_keys,
        "is_unique": duplicate_keys == 0
    })

pk_summary_df = pd.DataFrame(pk_audit)
print("\n Primary Key Uniqueness Audit ")
display(pk_summary_df)

 Full Row Duplicates Audit 


,dataset,total_rows,duplicate_rows,duplicate_pct
0,customers,99441,0,0.0000
1,geolocation,1000163,261831,26.1788
2,order_items,112650,0,0.0000
3,payments,103886,0,0.0000
4,reviews,99224,0,0.0000
5,orders,99441,0,0.0000
6,products,32951,0,0.0000
7,sellers,3095,0,0.0000
8,category_translation,71,0,0.0000



=== Primary Key Uniqueness Audit ===


,dataset,key_columns,total_records,unique_keys,duplicate_keys,is_unique
0,customers,customer_id,99441,99441,0,True
1,orders,order_id,99441,99441,0,True
2,order_items,"order_id, order_item_id",112650,112650,0,True
3,payments,"order_id, payment_sequential",103886,103886,0,True
4,reviews,review_id,99224,98410,814,False
5,products,product_id,32951,32951,0,True
6,sellers,seller_id,3095,3095,0,True
7,geolocation,geolocation_zip_code_prefix,1000163,19015,981148,False


In [6]:

# Step 4: Data Types & Timestamp Casting Audit

# Expected timestamp columns across Olist schema
expected_datetime_cols = {
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],
    "order_items": [
        "shipping_limit_date"
    ],
    "reviews": [
        "review_creation_date",
        "review_answer_timestamp"
    ]
}

dtype_audit = []

for table_name, cols in expected_datetime_cols.items():
    df = datasets[table_name]
    for col in cols:
        current_type = str(df[col].dtype)
        sample_val = str(df[col].dropna().iloc[0]) if not df[col].dropna().empty else "None"
        dtype_audit.append({
            "dataset": table_name,
            "column": col,
            "current_dtype": current_type,
            "expected_dtype": "datetime64[ns]",
            "sample_value": sample_val,
            "needs_conversion": current_type != "datetime64[ns]"
        })

dtype_summary_df = pd.DataFrame(dtype_audit)
print("=== Datetime Columns Audit ===")
display(dtype_summary_df)

# Zip Code format audit (Checking if stored as numeric vs string)
zip_audit = []
zip_cols = {
    "customers": "customer_zip_code_prefix",
    "sellers": "seller_zip_code_prefix",
    "geolocation": "geolocation_zip_code_prefix"
}

for table_name, col in zip_cols.items():
    df = datasets[table_name]
    current_type = str(df[col].dtype)
    sample_val = str(df[col].iloc[0])
    zip_audit.append({
        "dataset": table_name,
        "column": col,
        "current_dtype": current_type,
        "sample_value": sample_val,
        "is_string": current_type == "object"
    })

zip_summary_df = pd.DataFrame(zip_audit)
print("\n=== Zip Code Format Audit ===")
display(zip_summary_df)

=== Datetime Columns Audit ===


,dataset,column,current_dtype,expected_dtype,sample_value,needs_conversion
0,orders,order_purchase_timestamp,str,datetime64[ns],2017-10-02 10:56:33,True
1,orders,order_approved_at,str,datetime64[ns],2017-10-02 11:07:15,True
2,orders,order_delivered_carrier_date,str,datetime64[ns],2017-10-04 19:55:00,True
3,orders,order_delivered_customer_date,str,datetime64[ns],2017-10-10 21:25:13,True
4,orders,order_estimated_delivery_date,str,datetime64[ns],2017-10-18 00:00:00,True
5,order_items,shipping_limit_date,str,datetime64[ns],2017-09-19 09:45:35,True
6,reviews,review_creation_date,str,datetime64[ns],2018-01-18 00:00:00,True
7,reviews,review_answer_timestamp,str,datetime64[ns],2018-01-18 21:46:59,True



=== Zip Code Format Audit ===


,dataset,column,current_dtype,sample_value,is_string
0,customers,customer_zip_code_prefix,int64,14409,False
1,sellers,seller_zip_code_prefix,int64,13023,False
2,geolocation,geolocation_zip_code_prefix,int64,1037,False


In [7]:
# Step 5: Referential Integrity & Business Logic Sanity Audit

# 1. Foreign Key / Orphan Records Audit
order_ids_in_orders = set(orders["order_id"])
customer_ids_in_customers = set(customers["customer_id"])

orphan_order_items = (~order_items["order_id"].isin(order_ids_in_orders)).sum()
orphan_payments = (~payments["order_id"].isin(order_ids_in_orders)).sum()
orphan_reviews = (~reviews["order_id"].isin(order_ids_in_orders)).sum()
orphan_orders_customers = (~orders["customer_id"].isin(customer_ids_in_customers)).sum()

integrity_audit = pd.DataFrame([
    {"relationship": "order_items -> orders", "key": "order_id", "orphan_records": orphan_order_items},
    {"relationship": "payments -> orders", "key": "order_id", "orphan_records": orphan_payments},
    {"relationship": "reviews -> orders", "key": "order_id", "orphan_records": orphan_reviews},
    {"relationship": "orders -> customers", "key": "customer_id", "orphan_records": orphan_orders_customers}
])

print("=== Referential Integrity (Foreign Keys) ===")
display(integrity_audit)

# 2. Business Logic: Numeric Sanity (Price & Freight)
negative_prices = (order_items["price"] <= 0).sum()
negative_freight = (order_items["freight_value"] < 0).sum()
zero_payment_values = (payments["payment_value"] <= 0).sum()

# 3. Timeline Sanity (Delivered before purchased?)
orders_temp = orders.copy()
orders_temp["purchase_dt"] = pd.to_datetime(orders_temp["order_purchase_timestamp"], errors="coerce")
orders_temp["delivered_dt"] = pd.to_datetime(orders_temp["order_delivered_customer_date"], errors="coerce")

delivered_before_purchase = (orders_temp["delivered_dt"] < orders_temp["purchase_dt"]).sum()

logic_audit = pd.DataFrame([
    {"check": "Negative / Zero Prices in order_items", "violation_count": negative_prices},
    {"check": "Negative Freight Values in order_items", "violation_count": negative_freight},
    {"check": "Zero / Negative Payment Value in payments", "violation_count": zero_payment_values},
    {"check": "Delivered before Purchase Timestamp in orders", "violation_count": delivered_before_purchase}
])

print("\n=== Business Logic Sanity Checks ===")
display(logic_audit)

=== Referential Integrity (Foreign Keys) ===


,relationship,key,orphan_records
0,order_items -> orders,order_id,0
1,payments -> orders,order_id,0
2,reviews -> orders,order_id,0
3,orders -> customers,customer_id,0



=== Business Logic Sanity Checks ===


,check,violation_count
0,Negative / Zero Prices in order_items,0
1,Negative Freight Values in order_items,0
2,Zero / Negative Payment Value in payments,9
3,Delivered before Purchase Timestamp in orders,0


In [9]:
# Step 6: Master Audit Findings Summary

audit_findings = pd.DataFrame([
    {
        "Category": "Missing Values",
        "Issue Identified": "Orders timestamps (delivered/carrier) missing in ~3% rows; Products metadata missing in 610 rows; Reviews comments missing in 58-88% rows",
        "Impact": "Delivery metrics bias, missing category revenue attribution",
        "Cleaning Action Required": "Filter delivered orders for SLAs; impute missing categories as 'uncategorized'; preserve review scores"
    },
    {
        "Category": "Duplicates",
        "Issue Identified": "Geolocation has 261,831 full row duplicates; Reviews table has 814 duplicate review_ids",
        "Impact": "Memory wastage and Cartesian explosion during geo-merges; skewed review counts",
        "Cleaning Action Required": "Aggregate geolocation to unique zip prefixes; deduplicate reviews by keeping latest timestamp"
    },
    {
        "Category": "Data Types",
        "Issue Identified": "All 8 datetime columns stored as object/string; Zip codes stored as int64 (leading zero drop)",
        "Impact": "Date arithmetic failure; broken postal code matches",
        "Cleaning Action Required": "Convert all timestamps via pd.to_datetime(); zero-pad zip codes to 5-digit strings (.str.zfill(5))"
    },
    {
        "Category": "Business Logic",
        "Issue Identified": "9 payment transactions with value = 0",
        "Impact": "Minor GMV discrepancy",
        "Cleaning Action Required": "Investigate and filter out non-value transactions in payments cleaning"
    }
])

display(audit_findings)
print("\n SUCCESS audit complete!")

,Category,Issue Identified,Impact,Cleaning Action Required
0,Missing Values,Orders timestamps (delivered/carrier) missing in ~3% rows; Products metadata missing in 610 rows; Reviews comments missing in 58-88% rows,"Delivery metrics bias, missing category revenue attribution",Filter delivered orders for SLAs; impute missing categories as 'uncategorized'; preserve review scores
1,Duplicates,"Geolocation has 261,831 full row duplicates; Reviews table has 814 duplicate review_ids",Memory wastage and Cartesian explosion during geo-merges; skewed review counts,Aggregate geolocation to unique zip prefixes; deduplicate reviews by keeping latest timestamp
2,Data Types,All 8 datetime columns stored as object/string; Zip codes stored as int64 (leading zero drop),Date arithmetic failure; broken postal code matches,Convert all timestamps via pd.to_datetime(); zero-pad zip codes to 5-digit strings (.str.zfill(5))
3,Business Logic,9 payment transactions with value = 0,Minor GMV discrepancy,Investigate and filter out non-value transactions in payments cleaning



 SUCCESS audit complete!
